# NIT6003 — Session 4: Text Preprocessing
### A simple step-by-step notebook using discussion comments

This notebook extends our **Naive Bayes text-classification example**. We will clean the `Comment` column and keep the original text for comparison.

**Today's topics:** Loading data, URLs/emojis/emoticons, tokenisation, lowercasing, punctuation, stop words, stemming, lemmatisation, and final cleaned text. Classification is an optional final section.

**Note:** Some comments in the original dataset contain insulting language because the aim is to detect online insults.

## 0. Import the libraries

The first two NLTK downloads are **one-time setup commands**. Your Python environment must already have `pandas`, `nltk`, and `scikit-learn` installed.

In [1]:
import re
import string
import pandas as pd
import nltk
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

In [2]:
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package stopwords to /Users/thiru/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/thiru/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## 1. Load the dataset

`Comment` contains the text and `Insult` contains the class label (0 or 1).

The first line loads the **original Session 4 dataset**. If the class has no Internet access, **comment out the first line and uncomment the second** to use the bundled offline practice CSV, which has the same columns.

In [3]:
df = pd.read_csv("https://github.com/ipython-books/cookbook-2nd-data/blob/master/troll.csv?raw=true")
# df = pd.read_csv("Session4_Offline_Practice_Comments.csv")
df[["Insult", "Comment"]].tail()

,Insult,Comment
3942,1,"""you are both morons and that is never happening"""
3943,0,"""Many toolbars include spell check, like Yahoo..."
3944,0,"""@LambeauOrWrigley\xa0\xa0@K.Moss\xa0\nSioux F..."
3945,0,"""How about Felix? He is sure turning into one ..."
3946,0,"""You're all upset, defending this hipster band..."


In [4]:
print(df.shape)
print(df.columns)

(3947, 3)
Index(['Insult', 'Date', 'Comment'], dtype='object')


## 2. Remove URLs, emoticons and emojis

We do this **before removing punctuation**, because punctuation is part of a URL (`https://`) or an emoticon (`:)`). The original `Comment` column stays unchanged.

`.str.replace(..., regex=True)` finds a pattern and replaces it with a space.

In [5]:
df["no_noise"] = df["Comment"].str.replace(r"https?://\S+|www\.\S+", " ", regex=True)
df["no_noise"] = df["no_noise"].str.replace(r":-?\)|:-?\(|;\)|:D|:P|<3", " ", regex=True)
df["no_noise"] = df["no_noise"].str.replace(r"[\U0001F300-\U0001FAFF\u2600-\u27BF]", " ", regex=True)

df[["Comment", "no_noise"]].head(5)

,Comment,no_noise
0,"""You fuck your dad.""","""You fuck your dad."""
1,"""i really don't understand your point.\xa0 It ...","""i really don't understand your point.\xa0 It ..."
2,"""A\\xc2\\xa0majority of Canadians can and has ...","""A\\xc2\\xa0majority of Canadians can and has ..."
3,"""listen if you dont wanna get married to a man...","""listen if you dont wanna get married to a man..."
4,"""C\xe1c b\u1ea1n xu\u1ed1ng \u0111\u01b0\u1edd...","""C\xe1c b\u1ea1n xu\u1ed1ng \u0111\u01b0\u1edd..."


**Why this order?** If `https://example.com` is stripped of punctuation first, it becomes `httpsexamplecom`, which a URL pattern will not recognise.

*Practice:* Find a comment with a URL, emoji or emoticon. If none appears in the original discussion data, try the included offline practice CSV.

## 3. Tokenisation

Split each comment into words (tokens). We use `preserve_line=True` so NLTK does not require its separate sentence-tokeniser data.

In [6]:
df["tokens"] = df["no_noise"].apply(lambda text: word_tokenize(text, preserve_line=True))
df[["Comment", "tokens"]].tail(3)

,Comment,tokens
3944,"""@LambeauOrWrigley\xa0\xa0@K.Moss\xa0\nSioux F...","[``, @, LambeauOrWrigley\xa0\xa0, @, K.Moss\xa..."
3945,"""How about Felix? He is sure turning into one ...","[``, How, about, Felix, ?, He, is, sure, turni..."
3946,"""You're all upset, defending this hipster band...","[``, You, 're, all, upset, ,, defending, this,..."


## 4. Lowercasing

Convert every token to lowercase, for example `Happy` → `happy`.

In [7]:
df["lower_tokens"] = df["tokens"].apply(lambda words: [w.lower() for w in words])
df[["tokens", "lower_tokens"]].tail(3)

,tokens,lower_tokens
3944,"[``, @, LambeauOrWrigley\xa0\xa0, @, K.Moss\xa...","[``, @, lambeauorwrigley\xa0\xa0, @, k.moss\xa..."
3945,"[``, How, about, Felix, ?, He, is, sure, turni...","[``, how, about, felix, ?, he, is, sure, turni..."
3946,"[``, You, 're, all, upset, ,, defending, this,...","[``, you, 're, all, upset, ,, defending, this,..."


## 5. Remove punctuation

`string.punctuation` gives us characters such as `!`, `?`, `,` and `.`. `translate()` removes them from each token. The second line removes empty tokens.

In [8]:
punctuation = str.maketrans("", "", string.punctuation)
df["no_punct_tokens"] = df["lower_tokens"].apply(lambda words: [w.translate(punctuation) for w in words])
df["no_punct_tokens"] = df["no_punct_tokens"].apply(lambda words: [w for w in words if w])

df[["lower_tokens", "no_punct_tokens"]].head(3)

,lower_tokens,no_punct_tokens
0,"[``, you, fuck, your, dad, ., '']","[you, fuck, your, dad]"
1,"[``, i, really, do, n't, understand, your, poi...","[i, really, do, nt, understand, your, pointxa0..."
2,"[``, a\\xc2\\xa0majority, of, canadians, can, ...","[axc2xa0majority, of, canadians, can, and, has..."


## 6. Remove stop words

Stop words are frequent words like `the`, `and`, `is`. **Keep `no`, `not` and `never`**: they can change the meaning of a sentence.

Note: contractions such as `don't` need extra attention in a more advanced preprocessing pipeline.

In [9]:
stop_words = set(stopwords.words("english"))
stop_words = stop_words - {"no", "not", "never"}

df["filtered_tokens"] = df["no_punct_tokens"].apply(
    lambda words: [w for w in words if w not in stop_words]
)
df[["no_punct_tokens", "filtered_tokens"]].tail(3)

,no_punct_tokens,filtered_tokens
3944,"[lambeauorwrigleyxa0xa0, kmossxa0nsioux, falls...","[lambeauorwrigleyxa0xa0, kmossxa0nsioux, falls..."
3945,"[how, about, felix, he, is, sure, turning, int...","[felix, sure, turning, one, hell, starting, pi..."
3946,"[you, re, all, upset, defending, this, hipster...","[upset, defending, hipster, band, douches, rea..."


In [10]:
df["before_stopwords"] = df["no_punct_tokens"].apply(len)
df["after_stopwords"] = df["filtered_tokens"].apply(len)
df[["Comment", "before_stopwords", "after_stopwords"]].head(3)

,Comment,before_stopwords,after_stopwords
0,"""You fuck your dad.""",4,2
1,"""i really don't understand your point.\xa0 It ...",16,8
2,"""A\\xc2\\xa0majority of Canadians can and has ...",66,27


## 7. Stemming

Stemming removes word endings. A stem may **not** be a real dictionary word. We use Porter's stemmer.

In [11]:
porter = PorterStemmer()
df["stemmed_tokens"] = df["filtered_tokens"].apply(
    lambda words: [porter.stem(w) for w in words]
)
df[["filtered_tokens", "stemmed_tokens"]].tail(3)

,filtered_tokens,stemmed_tokens
3944,"[lambeauorwrigleyxa0xa0, kmossxa0nsioux, falls...","[lambeauorwrigleyxa0xa0, kmossxa0nsioux, fall,..."
3945,"[felix, sure, turning, one, hell, starting, pi...","[felix, sure, turn, one, hell, start, pitcher,..."
3946,"[upset, defending, hipster, band, douches, rea...","[upset, defend, hipster, band, douch, read, ne..."


## 8. Lemmatisation

Lemmatisation attempts to return a dictionary word (lemma). We use WordNet.

**Important:** Stemming and lemmatisation both start from `filtered_tokens` — do not lemmatise the stems.

In [12]:
lemmatizer = WordNetLemmatizer()
df["lemmatised_tokens"] = df["filtered_tokens"].apply(
    lambda words: [lemmatizer.lemmatize(w) for w in words]
)
df[["filtered_tokens", "lemmatised_tokens"]].tail(3)

,filtered_tokens,lemmatised_tokens
3944,"[lambeauorwrigleyxa0xa0, kmossxa0nsioux, falls...","[lambeauorwrigleyxa0xa0, kmossxa0nsioux, fall,..."
3945,"[felix, sure, turning, one, hell, starting, pi...","[felix, sure, turning, one, hell, starting, pi..."
3946,"[upset, defending, hipster, band, douches, rea...","[upset, defending, hipster, band, douche, read..."


WordNet's basic `lemmatize()` call treats words as nouns by default. More advanced lemmatisation can use part-of-speech tags, but we will keep it simple here.

## 9. Compare stems and lemmas

Look at the same words before and after stemming/lemmatisation. Change `0` to another row number if you want a different comment.

In [19]:
pd.DataFrame({
    "original": df.loc[3944, "filtered_tokens"],
    "stem": df.loc[3944, "stemmed_tokens"],
    "lemma": df.loc[3944, "lemmatised_tokens"]
}).tail(10)

,original,stem,lemma
2,falls,fall,fall
3,sd,sd,sd
4,told,told,told
5,boy,boy,boy
6,call,call,call
7,around,around,around
8,today,today,today
9,figure,figur,figure
10,options,option,option
11,hope,hope,hope


## 10. Create the final cleaned text

Join the **lemmatised tokens**, separated by spaces. Since social-media noise was removed at the start, we do not need to clean it a second time.

In [14]:
df["final_clean_text"] = df["lemmatised_tokens"].apply(" ".join)
df[["Comment", "final_clean_text"]].tail(5)

,Comment,final_clean_text
3942,"""you are both morons and that is never happening""",moron never happening
3943,"""Many toolbars include spell check, like Yahoo...",many toolbars include spell check like yahoo e...
3944,"""@LambeauOrWrigley\xa0\xa0@K.Moss\xa0\nSioux F...",lambeauorwrigleyxa0xa0 kmossxa0nsioux fall sd ...
3945,"""How about Felix? He is sure turning into one ...",felix sure turning one hell starting pitcher t...
3946,"""You're all upset, defending this hipster band...",upset defending hipster band douche reading ne...


## 11. Review each preprocessing stage

Every stage has its own column, so we can explain exactly what was changed. You can display any two columns together.

In [15]:
df[["Comment", "no_noise", "tokens", "lower_tokens", "no_punct_tokens",
    "filtered_tokens", "stemmed_tokens", "lemmatised_tokens", "final_clean_text"]].tail(3)

,Comment,no_noise,tokens,lower_tokens,no_punct_tokens,filtered_tokens,stemmed_tokens,lemmatised_tokens,final_clean_text
3944,"""@LambeauOrWrigley\xa0\xa0@K.Moss\xa0\nSioux F...","""@LambeauOrWrigley\xa0\xa0@K.Moss\xa0\nSioux F...","[``, @, LambeauOrWrigley\xa0\xa0, @, K.Moss\xa...","[``, @, lambeauorwrigley\xa0\xa0, @, k.moss\xa...","[lambeauorwrigleyxa0xa0, kmossxa0nsioux, falls...","[lambeauorwrigleyxa0xa0, kmossxa0nsioux, falls...","[lambeauorwrigleyxa0xa0, kmossxa0nsioux, fall,...","[lambeauorwrigleyxa0xa0, kmossxa0nsioux, fall,...",lambeauorwrigleyxa0xa0 kmossxa0nsioux fall sd ...
3945,"""How about Felix? He is sure turning into one ...","""How about Felix? He is sure turning into one ...","[``, How, about, Felix, ?, He, is, sure, turni...","[``, how, about, felix, ?, he, is, sure, turni...","[how, about, felix, he, is, sure, turning, int...","[felix, sure, turning, one, hell, starting, pi...","[felix, sure, turn, one, hell, start, pitcher,...","[felix, sure, turning, one, hell, starting, pi...",felix sure turning one hell starting pitcher t...
3946,"""You're all upset, defending this hipster band...","""You're all upset, defending this hipster band...","[``, You, 're, all, upset, ,, defending, this,...","[``, you, 're, all, upset, ,, defending, this,...","[you, re, all, upset, defending, this, hipster...","[upset, defending, hipster, band, douches, rea...","[upset, defend, hipster, band, douch, read, ne...","[upset, defending, hipster, band, douche, read...",upset defending hipster band douche reading ne...


### Student practice — discuss before the assessment

1. Show three examples of tokenisation, punctuation removal and stop-word removal.
2. Find two words where stemming produces a shortened form that is not a dictionary word.
3. Find three words and compare their original form, stem and lemma.
4. Explain one disadvantage of removing punctuation, stop words, emojis or emoticons.
5. Why must we preserve the original `Comment` column?

**Independent assessment:** You will use a *different CSV and text column*. Adapt these techniques to the assessment instructions; do not simply copy the original dataset/column names. The assessment prohibits generative AI use during the supervised test.

---
## Optional: Text classification with TF-IDF and Naive Bayes

This final section returns to the **original Session 4 topic**. You can skip it when preparing specifically for the preprocessing assessment.

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import BernoulliNB

X_train, X_test, y_train, y_test = train_test_split(
    df["final_clean_text"], df["Insult"], test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

In [17]:
model = BernoulliNB()
model.fit(X_train, y_train)
print("Accuracy:", model.score(X_test, y_test))

Accuracy: 0.7341772151898734


### Reference

Adapted from the original Session 4 notebook: [IPython Cookbook — Learning from text: Naive Bayes for NLP](https://ipython-books.github.io/84-learning-from-text-naive-bayes-for-natural-language-processing/).